# Cross-Sectional Stock Return Prediction

This notebook runs the attention experiment from `run_project_mhattn.py`.
This fixed-seed notebook is kept with outputs for reproducible single-seed comparison.


In [1]:
import importlib
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython import get_ipython

ip = get_ipython()
if ip is not None:
    ip.run_line_magic("matplotlib", "inline")
    try:
        from matplotlib_inline.backend_inline import set_matplotlib_formats
        set_matplotlib_formats("png")
    except Exception:
        pass

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import stock_return_core as srp
import run_project_mhattn as mh

srp = importlib.reload(srp)
mh = importlib.reload(mh)
print(srp.__file__)
print(mh.__file__)


/home/hlc/ECE_C147A_Project/stock_return_project4.py
/home/hlc/ECE_C147A_Project/run_project_mhattn.py


In [2]:
# Quick run defaults for the seeded attention experiment.
START_DATE = "2015-01-01"
END_DATE = "2025-01-01"
LOOKBACK = 60
HORIZON = 5
UNIVERSE = "small"  # small | sp500 | nasdaq100 | auto
MODELS = ["RNN", "LSTM", "GRU", "TRANSFORMER"]
ATTN_HEADS = 4
DEVICE = mh.resolve_device()
SEED = 42

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = True

UNIVERSE_TO_TICKERS = {
    "small": srp.DEFAULT_LIQUID_TICKERS,
    "sp500": srp.SP500_TICKERS,
    "nasdaq100": srp.NASDAQ100_TICKERS,
    "auto": None,
}
TICKERS = UNIVERSE_TO_TICKERS[UNIVERSE]
print(f"Universe: {UNIVERSE}")
print(f"Seed: {SEED}")

# Universe filter: keep stocks that cover nearly the full sample window.
START_BUFFER_DAYS = 5
END_BUFFER_DAYS = 5
MIN_COVERAGE_RATIO = 0.95


Universe: small
Seed: 42


In [3]:
raw_prices = srp.download_price_history(start=START_DATE, end=END_DATE, tickers=TICKERS)

flat_prices = srp.flatten_price_frame(raw_prices).dropna(subset=["Close"]).copy()
all_dates = pd.Index(sorted(pd.to_datetime(flat_prices["Date"].unique())))
date_to_pos = {date: idx for idx, date in enumerate(all_dates)}

coverage = (
    flat_prices.groupby("Ticker")
    .agg(
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        n_dates=("Date", "nunique"),
    )
    .reset_index()
)
coverage["first_pos"] = pd.to_datetime(coverage["first_date"]).map(date_to_pos)
coverage["last_pos"] = pd.to_datetime(coverage["last_date"]).map(date_to_pos)
coverage["coverage_ratio"] = coverage["n_dates"] / len(all_dates)

eligible_tickers = coverage.loc[
    (coverage["first_pos"] <= START_BUFFER_DAYS)
    & (coverage["last_pos"] >= len(all_dates) - 1 - END_BUFFER_DAYS)
    & (coverage["coverage_ratio"] >= MIN_COVERAGE_RATIO),
    "Ticker",
].tolist()

filtered_prices = raw_prices.loc[
    :,
    raw_prices.columns.get_level_values(0).isin(eligible_tickers),
]

print(
    f"Kept {len(eligible_tickers)} / {coverage.shape[0]} tickers "
    f"with >= {MIN_COVERAGE_RATIO:.0%} coverage and near-full sample history."
)
print(
    "Dropped examples:",
    coverage.loc[~coverage["Ticker"].isin(eligible_tickers), "Ticker"].head(10).tolist(),
)

experiment_data = srp.prepare_experiment_data(
    filtered_prices,
    horizon=HORIZON,
    lookback=LOOKBACK,
    train_size=0.7,
    val_size=0.15,
)
for split_name, split_df in experiment_data.splits.items():
    print(split_name, split_df["Date"].min(), split_df["Date"].max(), len(split_df))


Kept 49 / 50 tickers with >= 95% coverage and near-full sample history.
Dropped examples: ['UBER']


train 2015-02-02 00:00:00 2021-12-31 00:00:00 85407
val 2022-01-03 00:00:00 2023-06-29 00:00:00 18326
test 2023-06-30 00:00:00 2024-12-23 00:00:00 18326


In [4]:
MODEL_CONFIGS = {
    "RNN": srp.TrainConfig(
        batch_size=128,
        hidden_dim=64,
        num_layers=1,
        dropout=0.0,
        learning_rate=1e-4,
        weight_decay=1e-5,
        max_epochs=20,
        patience=6,
        grad_clip=1.0,
        device=DEVICE,
    ),
    "LSTM": srp.TrainConfig(
        batch_size=128,
        hidden_dim=64,
        num_layers=2,
        dropout=0.15,
        learning_rate=3e-4,
        weight_decay=1e-5,
        max_epochs=25,
        patience=6,
        grad_clip=1.0,
        device=DEVICE,
    ),
    "GRU": srp.TrainConfig(
        batch_size=128,
        hidden_dim=64,
        num_layers=2,
        dropout=0.15,
        learning_rate=1e-4,
        weight_decay=1e-5,
        max_epochs=25,
        patience=6,
        grad_clip=1.0,
        device=DEVICE,
    ),
    "TRANSFORMER": srp.TrainConfig(
        batch_size=256,
        hidden_dim=160,
        num_layers=3,
        dropout=0.15,
        num_heads=8,
        learning_rate=3e-4,
        weight_decay=5e-5,
        max_epochs=35,
        patience=10,
        grad_clip=1.0,
        device=DEVICE,
    ),
}
input_dim = len(experiment_data.feature_columns)
results = {}
for model_name in MODELS:
    train_config = MODEL_CONFIGS[model_name]
    set_seed(SEED)
    print(f"=== Notebook model: {model_name} (seed={SEED}) ===")
    if model_name == "RNN":
        model = srp.build_model(
            model_name="RNN",
            input_dim=input_dim,
            hidden_dim=train_config.hidden_dim,
            num_layers=train_config.num_layers,
            dropout=train_config.dropout,
        )
    elif model_name in {"LSTM", "GRU"}:
        model = mh.AttentionSequenceRegressor(
            input_dim=input_dim,
            hidden_dim=train_config.hidden_dim,
            num_layers=train_config.num_layers,
            dropout=train_config.dropout,
            cell_type=model_name,
            num_attn_heads=ATTN_HEADS,
        )
    elif model_name == "TRANSFORMER":
        model = srp.build_model(
            model_name="TRANSFORMER",
            input_dim=input_dim,
            hidden_dim=train_config.hidden_dim,
            num_layers=train_config.num_layers,
            dropout=train_config.dropout,
            num_heads=train_config.num_heads,
        )
    else:
        raise ValueError(f"Unsupported model: {model_name}")
    results[model_name] = mh.run_experiment(
        model,
        model_name,
        experiment_data,
        train_config,
    )


=== Notebook model: RNN (seed=42) ===

=== Training RNN ===


[train] batch 1/645 - loss: 0.247352


[train] batch 200/645 - loss: 0.021102


[train] batch 400/645 - loss: 0.011883


[train] batch 600/645 - loss: 0.008643


...

[train] batch 323/323 - loss: 0.001554


[val] batch 1/61 - loss: 0.001913


[val] batch 61/61 - loss: 0.002154


Epoch 35/35 - train_loss: 0.001554, val_loss: 0.002154, time: 18.8s


No improvement. Early-stop patience left: 2


In [5]:
summary = srp.build_summary_frame(results)
summary


,model,mean_ic,std_ic,ic_t_stat,mean_rank_ic,std_rank_ic,rank_ic_t_stat,mean_return,volatility,sharpe_ratio
0,TRANSFORMER,0.081623,0.313623,4.611781,0.055979,0.285949,3.468983,0.007258,0.026972,4.271542
1,LSTM,0.067292,0.277998,4.289335,0.046877,0.250773,3.312428,0.006698,0.024165,4.400267
2,GRU,0.066983,0.250178,4.744425,0.052918,0.244788,3.830688,0.006469,0.023143,4.437182
3,RNN,0.018729,0.201266,1.648936,0.016397,0.185735,1.564329,0.001392,0.019289,1.145264


# Result Visualizations

Plots for loss curves, IC/RankIC, portfolio performance, and model comparison.


In [6]:
import matplotlib.pyplot as plt

best_model = summary.iloc[0]["model"]
print("Visualizing model:", best_model)
training_fig = srp.plot_training_histories(results)
display(training_fig)
plt.close(training_fig)


Visualizing model: TRANSFORMER


<Figure size 1000x1400 with 4 Axes>

In [7]:
import matplotlib.pyplot as plt

ic_fig = srp.plot_ic_series(results, best_model, split="test", rolling_window=20)
display(ic_fig)
plt.close(ic_fig)


<Figure size 1200x700 with 2 Axes>

In [8]:
import matplotlib.pyplot as plt

for model_name, result in results.items():
    if "portfolio_curve" not in result:
        portfolio_curve, portfolio_summary = srp.backtest_long_short(result["test_predictions"])
        result["portfolio_curve"] = portfolio_curve
        result["portfolio_summary"] = portfolio_summary

portfolio_fig = srp.plot_portfolio_curve(results, best_model)
summary_fig = srp.plot_summary_bars(summary)
display(portfolio_fig)
display(summary_fig)
plt.close(portfolio_fig)
plt.close(summary_fig)


<Figure size 1200x450 with 1 Axes>

<Figure size 1500x400 with 3 Axes>

# Prediction diagnostics

Inspect raw predictions, sign accuracy, and the prediction-vs-target scatter plot.


In [9]:
model_name = best_model
pred = results[model_name]["test_predictions"].copy()
pred.head()


,Date,Ticker,prediction,target
0,2023-09-26,AAPL,0.006715,0.002559
1,2023-09-26,ABBV,0.004678,-0.042103
2,2023-09-26,ABT,0.004233,-0.004572
3,2023-09-26,ADBE,0.005884,0.001442
4,2023-09-26,AMD,0.011096,0.042935


In [10]:
pred["pred_sign"] = (pred["prediction"] > 0).astype(int)
pred["true_sign"] = (pred["target"] > 0).astype(int)

sign_accuracy = (pred["pred_sign"] == pred["true_sign"]).mean()
daily_sign_accuracy = pred.groupby("Date").apply(
    lambda x: ((x["prediction"] > 0) == (x["target"] > 0)).mean(),
    include_groups=False,
)

print("Overall sign accuracy:", round(float(sign_accuracy), 4))
daily_sign_accuracy.describe()


Overall sign accuracy: 0.5656


count    314.000000
mean       0.565579
std        0.207496
min        0.081633
25%        0.428571
50%        0.571429
75%        0.734694
max        1.000000
dtype: float64

In [11]:
import matplotlib.pyplot as plt

latest_day = pred["Date"].max()
latest_slice = pred[pred["Date"] == latest_day].copy()
display(latest_slice.sort_values("prediction", ascending=False).head(10))

ax = latest_slice.plot.scatter(
    x="prediction",
    y="target",
    figsize=(6, 5),
    alpha=0.7,
    title=f"{model_name} prediction vs target on {latest_day.date()}"
)
ax.axhline(0.0, color="black", linewidth=1, alpha=0.6)
ax.axvline(0.0, color="black", linewidth=1, alpha=0.6)
plt.show()


,Date,Ticker,prediction,target,pred_sign,true_sign
15372,2024-12-23,NVDA,0.009607,-0.038519,1,0
15341,2024-12-23,AMD,0.008912,-0.030578,1,0
15340,2024-12-23,ADBE,0.008847,-0.004611,1,0
15381,2024-12-23,UNH,0.008291,-0.000948,1,0
15367,2024-12-23,META,0.007984,-0.023906,1,0
15343,2024-12-23,AMZN,0.007958,-0.025193,1,0
15371,2024-12-23,NOW,0.007916,-0.025858,1,0
15356,2024-12-23,GOOGL,0.007643,-0.027385,1,0
15370,2024-12-23,NFLX,0.007624,-0.022086,1,0
15344,2024-12-23,AVGO,0.007593,-0.002195,1,0
